# OpenRouter LLM annotacija članaka

Ova verzija je očišćena od Ollama koda i koristi samo OpenRouter API.

Šta notebook radi:
1. učita CSV dataset,
2. pošalje članke na OpenRouter model,
3. traži striktan JSON output,
4. validira JSON,
5. snimi raw rezultate, flat CSV i agreement summary.

Napomena: kolona `tema` u datasetu je samo početna/sampling klasa i ne šalje se modelu u prompt, da ne bi biasirala anotaciju.


In [2]:
# 1) OPENROUTER API KEY
# Opcija A: pokreni ćeliju i unesi key kada te pita.
# Opcija B: otkomentariši liniju ispod i ručno upiši key, ali to nije preporučeno ako dijeliš notebook.
# os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-..."

import os
from getpass import getpass

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass("Unesi OPENROUTER_API_KEY: ").strip()
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY nije postavljen.")

print("OPENROUTER_API_KEY je postavljen.")


OPENROUTER_API_KEY je postavljen.


In [5]:
# 2) IMPORTI

import json
import os
import re
import time
from pathlib import Path

import pandas as pd
import requests


In [15]:
# 3) KONFIGURACIJA

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Model ID na OpenRouteru.
# Možeš promijeniti npr. u "qwen/qwen3.6-max-preview" ako želiš jači model.
MODEL = "qwen/qwen3.6-35b-a3b"

# Ako je None, notebook pokušava automatski pronaći CSV u trenutnom folderu, /content ili /mnt/data.
# Ako radiš u Colabu, uploaduj CSV u Files panel i ostavi None ili upiši tačan path.
INPUT_CSV = None

OUTPUT_DIR = "openrouter_prompt_tests"

# Za test ostavi 5/10/50. Za cijeli dataset stavi None.
N_ROWS = None

# Samo prompt 2 (evidence-first) = len(df) * 1 API poziv.
PROMPT_NAMES_TO_RUN = [
    "prompt_2_evidence_first",
]

SLEEP_BETWEEN = 0.2
MAX_RETRIES = 3
RETRY_BACKOFF = 5

# Ako se batch prekine, ponovni run preskače već obrađene (article_id, prompt_name).
RESUME = True

# Kod Qwen reasoning modela ovo smanjuje trošak i šansu da model vrati <think>.
# Ako OpenRouter/provider ne prihvati ovaj parametar, kod automatski ponovi request bez njega.
DISABLE_REASONING = True

GENERATION_OPTIONS = {
    "temperature": 0,
    "top_p": 0.9,
    "max_tokens": 2000,
}

TOPIC_KEYS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]


In [16]:
df = pd.read_csv("novi_sample_za_llm.csv", dtype={"article_id": str})

required_columns = ["PORTAL", "DATUM", "NASLOV", "SADRZAJ", "article_id"]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Nedostaju obavezne kolone u CSV-u: {missing}")

df["SADRZAJ"] = df["SADRZAJ"].fillna("")

if "article_id" not in df.columns:
    df["article_id"] = range(len(df))
df["article_id"] = df["article_id"].astype(str)

print("Shape:", df.shape)
display(df.head(3))
print("\nNaN po kolonama:")
display(df.isna().sum().sort_values(ascending=False).head(15))


Shape: (4000, 9)


,PORTAL,DATUM,NASLOV,SADRZAJ,euroatlantske_integracije_mentioned,izborna_reforma_mentioned,negiranje_genocida_mentioned,gradjanska_vs_konstitutivni_mentioned,article_id
0,Klix.ba,2022-07-25,Evo šta su građani poručili sa današnjih prote...,Više od 7.000 građana okupilo se na protestima...,1,1,0,1,12177
1,Klix.ba,2021-10-21,Čavara brani neodbranjivo: Dodik ne ruši držav...,Predsjednik Federacije BiH Marinko Čavara (HDZ...,0,0,1,1,21123
2,Klix.ba,2023-02-23,Statut Grada Mostara ni danas nije usvojen: Vi...,Gradsko vijeće Grada Mostara nakon iscrpne ras...,0,1,0,1,5583



NaN po kolonama:


PORTAL                                   0
DATUM                                    0
NASLOV                                   0
SADRZAJ                                  0
euroatlantske_integracije_mentioned      0
izborna_reforma_mentioned                0
negiranje_genocida_mentioned             0
gradjanska_vs_konstitutivni_mentioned    0
article_id                               0
dtype: int64

In [17]:
df

,PORTAL,DATUM,NASLOV,SADRZAJ,euroatlantske_integracije_mentioned,izborna_reforma_mentioned,negiranje_genocida_mentioned,gradjanska_vs_konstitutivni_mentioned,article_id
0,Klix.ba,2022-07-25,Evo šta su građani poručili sa današnjih prote...,Više od 7.000 građana okupilo se na protestima...,1,1,0,1,12177
1,Klix.ba,2021-10-21,Čavara brani neodbranjivo: Dodik ne ruši držav...,Predsjednik Federacije BiH Marinko Čavara (HDZ...,0,0,1,1,21123
2,Klix.ba,2023-02-23,Statut Grada Mostara ni danas nije usvojen: Vi...,Gradsko vijeće Grada Mostara nakon iscrpne ras...,0,1,0,1,5583
3,Klix.ba,2022-02-01,HDZ u Hrvatskoj raspravljao o izbornoj reformi...,Vrh vladajućeg HDZ-a u Hrvatskoj jučer je rasp...,0,1,0,1,17855
4,Klix.ba,2020-07-30,Hakeri postavljaju lažne priče na web portalim...,Hakeri su provalili u sisteme nekoliko stvarni...,1,0,0,0,34982
...,...,...,...,...,...,...,...,...,...
3995,Klix.ba,2023-01-17,Čović o prvom testu rada sa Osmorkom: Formiran...,Predsjednik HDZ-a Dragan Čović govorio je veče...,1,1,0,1,7023
3996,Klix.ba,2021-07-06,Predsjednik Hrvatske Zoran Milanović dolazi u ...,Predsjednik Republike Hrvatske Zoran Milanović...,0,0,1,1,24350
3997,Klix.ba,2022-11-10,"Milanović poručio Tužilaštvu BiH: Meni mogu ""p...",Nakon vijesti da Tužilaštvo BiH provjerava nav...,0,0,1,0,9106
3998,Klix.ba,2021-11-12,"Šarović: Dodikov put od 'titana' do patuljka, ...","Mirko Šarović, predsjednik Srpske demokratske ...",0,0,0,1,20361


In [18]:
# 5) PROMPTI
# Bitno: ne koristimo Python .format() jer JSON schema sadrži obične { } zagrade.
# Umjesto toga koristimo sigurnu zamjenu placeholdera tipa <<ARTICLE_ID>>.
# Napomena: koristimo samo prompt 2 (evidence_first). Prompt 1 i 3 su uklonjeni.

PROMPT_2 = """
Ti si strogi annotator političkih tema u novinskim člancima iz Bosne i Hercegovine.

Analiziraj članak i označi samo teme za koje postoji jasan tekstualni dokaz.

Pravila:
- Ne zaključuj na osnovu portala, autora ili političkih aktera ako tema nije stvarno prisutna u tekstu.
- Ne koristi svoje vanjsko znanje.
- Ne označavaj temu samo zato što se spominje političar.
- Za svaku temu koja je mentioned=true, moraš navesti 1 do 3 kratka evidence citata ili fraze iz članka.
- Ako nema citata/fraze koja podržava oznaku, mentioned mora biti false.
- Vrati samo JSON.

Teme i stance:

1. euroatlantske_integracije
Relevantno ako članak govori o EU, NATO, kandidatskom statusu, evropskom putu, MAP/ANP, integracijama, sankcijama ili geopolitici direktno vezanoj za EU/NATO put BiH.
stance:
1 = pro EU/NATO
0 = neutralno/miješano/nejasno
-1 = anti EU/NATO

2. negiranje_genocida
Relevantno ako članak govori o genocidu u Srebrenici, negiranju/minimiziranju genocida, presudama, ratnim zločincima, Inzkovom zakonu ili zabrani negiranja.
stance:
1 = priznaje/štiti istinu o genocidu ili osuđuje negiranje
0 = neutralno izvještava
-1 = negira/minimizira/relativizira genocid

3. gradjanska_vs_konstitutivni
Relevantno ako članak govori o građanskoj državi, konstitutivnim narodima, legitimnom predstavljanju, majorizaciji, etničkom predstavljanju, Domu naroda, ustavnoj strukturi BiH.
stance:
1 = pro građanski model
0 = neutralno/miješano/nejasno
-1 = pro konstitutivni narodi / legitimno predstavljanje

4. izborna_reforma
Relevantno ako članak govori o izmjenama Izbornog zakona, CIK-u, tehničkim izmjenama izbora, Schmidtovim/OHR izmjenama, pregovorima SDA-HDZ-DF, Ljubić presudi, Domu naroda u kontekstu izbora.
stance:
1 = podržava reformu/prijedlog
0 = neutralno/miješano/nejasno
-1 = protivi se reformi/prijedlogu

JSON mora imati tačno ovaj oblik:
{
  "article_id": "<<ARTICLE_ID>>",
  "is_relevant": false,
  "primary_topic": "none",
  "topics": {
    "euroatlantske_integracije": {
      "mentioned": false,
      "stance": 0,
      "confidence": 0.0,
      "evidence": []
    },
    "negiranje_genocida": {
      "mentioned": false,
      "stance": 0,
      "confidence": 0.0,
      "evidence": []
    },
    "gradjanska_vs_konstitutivni": {
      "mentioned": false,
      "stance": 0,
      "confidence": 0.0,
      "evidence": []
    },
    "izborna_reforma": {
      "mentioned": false,
      "stance": 0,
      "confidence": 0.0,
      "evidence": []
    }
  }
}

Sada popuni JSON za članak:

ID:
<<ARTICLE_ID>>

NASLOV:
<<NASLOV>>

SADRŽAJ:
<<SADRZAJ>>
"""


PROMPTS = {
    "prompt_2_evidence_first": PROMPT_2,
}


In [19]:
# 6) HELPERI ZA TEKST I PROMPT

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def truncate_text(text, max_chars=12000):
    """
    Ako je članak predug, šaljemo samo početak teksta.
    Kod vijesti su najvažnije informacije najčešće na početku.
    """
    text = clean_text(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n\n[TEKST SKRAĆEN ZBOG DUŽINE]"


def render_prompt(template, row):
    """
    Sigurnija zamjena od .format(), jer prompt sadrži JSON zagrade { }.
    """
    replacements = {
        "<<ARTICLE_ID>>": clean_text(row.get("article_id", "")),
        "<<PORTAL>>": clean_text(row.get("PORTAL", "")),
        "<<DATUM>>": clean_text(row.get("DATUM", "")),
        "<<NASLOV>>": clean_text(row.get("NASLOV", "")),
        "<<SADRZAJ>>": truncate_text(row.get("SADRZAJ", "")),
    }

    prompt = template
    for key, value in replacements.items():
        prompt = prompt.replace(key, value)

    return prompt


# Brzi sanity check da prompt render ne puca zbog JSON zagrada.
_test_prompt = render_prompt(PROMPTS["prompt_2_evidence_first"], df.iloc[0])
print(_test_prompt[:800])



Ti si strogi annotator političkih tema u novinskim člancima iz Bosne i Hercegovine.

Analiziraj članak i označi samo teme za koje postoji jasan tekstualni dokaz.

Pravila:
- Ne zaključuj na osnovu portala, autora ili političkih aktera ako tema nije stvarno prisutna u tekstu.
- Ne koristi svoje vanjsko znanje.
- Ne označavaj temu samo zato što se spominje političar.
- Za svaku temu koja je mentioned=true, moraš navesti 1 do 3 kratka evidence citata ili fraze iz članka.
- Ako nema citata/fraze koja podržava oznaku, mentioned mora biti false.
- Vrati samo JSON.

Teme i stance:

1. euroatlantske_integracije
Relevantno ako članak govori o EU, NATO, kandidatskom statusu, evropskom putu, MAP/ANP, integracijama, sankcijama ili geopolitici direktno vezanoj za EU/NATO put BiH.
stance:
1 = pro EU/NA


In [20]:
# 7) OPENROUTER POZIV

def call_openrouter(prompt, response_format=True, disable_reasoning=DISABLE_REASONING):
    """
    Poziva OpenRouter Chat Completions endpoint.
    Ako provider odbije reasoning ili response_format parametar, kod proba fallback.
    """

    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY nije postavljen.")

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    "Ti si strogi annotator. Vrati samo validan JSON koji odgovara traženoj shemi. "
                    "Ne piši markdown, objašnjenja, niti dodatni tekst."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        **GENERATION_OPTIONS,
    }

    if response_format:
        payload["response_format"] = {"type": "json_object"}

    if disable_reasoning:
        payload["reasoning"] = {"effort": "none", "exclude": True}

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://localhost",
        "X-Title": "bih-politicke-teme-annotacija",
    }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                OPENROUTER_URL,
                headers=headers,
                json=payload,
                timeout=120,
            )

            # Ako neki provider ne prihvati reasoning parametar, probaj bez njega.
            if response.status_code in {400, 422} and "reasoning" in payload:
                payload = dict(payload)
                payload.pop("reasoning", None)
                print("  Provider nije prihvatio reasoning parametar; ponavljam bez reasoning parametra.")
                continue

            # Ako neki provider ne prihvati response_format, probaj bez njega.
            if response.status_code in {400, 422} and "response_format" in payload:
                payload = dict(payload)
                payload.pop("response_format", None)
                print("  Provider nije prihvatio response_format; ponavljam bez response_format.")
                continue

            if response.status_code == 429:
                wait = RETRY_BACKOFF * attempt
                print(f"  Rate limit 429; čekam {wait}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()
            data = response.json()

            message = data["choices"][0]["message"]
            content = message.get("content", "")

            if not content:
                raise RuntimeError(f"OpenRouter response nema content. Full response: {data}")

            return content

        except requests.exceptions.RequestException as e:
            last_error = e
            wait = RETRY_BACKOFF * attempt
            print(f"  Request greška: {e}; pokušaj {attempt}/{MAX_RETRIES}; čekam {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"Svi pokušaji su pali. Posljednja greška: {last_error}")


In [21]:
# 8) JSON PARSIRANJE I VALIDACIJA

def extract_json(text):
    """
    Sa response_format=json_object model bi trebao vratiti čist JSON.
    Fallback: izvlačimo prvi JSON objekat iz teksta.
    """
    text = clean_text(text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("Nije pronađen JSON objekat u odgovoru modela.")

    return json.loads(match.group(0))


def validate_annotation(obj):
    errors = []

    if not isinstance(obj, dict):
        return False, ["Output nije dict."]

    if "article_id" not in obj:
        errors.append("Nedostaje article_id.")

    if "is_relevant" not in obj or not isinstance(obj["is_relevant"], bool):
        errors.append("Nedostaje ili je neispravan is_relevant.")

    allowed_primary = TOPIC_KEYS + ["none"]
    if obj.get("primary_topic") not in allowed_primary:
        errors.append(f"Neispravan primary_topic: {obj.get('primary_topic')}")

    topics = obj.get("topics")
    if not isinstance(topics, dict):
        errors.append("Nedostaje ili je neispravan topics.")
        return False, errors

    for topic in TOPIC_KEYS:
        if topic not in topics:
            errors.append(f"Nedostaje topic: {topic}")
            continue

        item = topics[topic]

        if not isinstance(item, dict):
            errors.append(f"{topic} nije dict.")
            continue

        if "mentioned" not in item or not isinstance(item["mentioned"], bool):
            errors.append(f"{topic}.mentioned nije bool.")

        if item.get("stance") not in [-1, 0, 1]:
            errors.append(f"{topic}.stance nije -1/0/1: {item.get('stance')}")

        conf = item.get("confidence")
        if not isinstance(conf, (int, float)) or not (0 <= conf <= 1):
            errors.append(f"{topic}.confidence nije u [0, 1]: {conf}")

        if "evidence" not in item or not isinstance(item["evidence"], list):
            errors.append(f"{topic}.evidence nije lista.")

        if item.get("mentioned") is False and item.get("stance", 0) != 0:
            errors.append(f"{topic}: mentioned=false, ali stance nije 0.")

        if item.get("mentioned") is True and item.get("confidence", 0) == 0:
            errors.append(f"{topic}: mentioned=true, ali confidence=0.")

    return len(errors) == 0, errors


def flatten_result(article_id, prompt_name, parsed, valid, errors, raw_response, source_row=None):
    source_row = source_row if source_row is not None else {}

    row = {
        "article_id": article_id,
        "prompt_name": prompt_name,
        "valid_json": bool(valid),
        "error": "; ".join(errors) if isinstance(errors, list) else str(errors),
        "raw_response": raw_response,
        "portal": clean_text(source_row.get("PORTAL", "")) if hasattr(source_row, "get") else "",
        "datum": clean_text(source_row.get("DATUM", "")) if hasattr(source_row, "get") else "",
        "naslov": clean_text(source_row.get("NASLOV", "")) if hasattr(source_row, "get") else "",
        "tema_sampling": clean_text(source_row.get("tema", "")) if hasattr(source_row, "get") else "",
    }

    if isinstance(parsed, dict):
        row["is_relevant"] = parsed.get("is_relevant")
        row["primary_topic"] = parsed.get("primary_topic")
        topics = parsed.get("topics", {})
    else:
        row["is_relevant"] = None
        row["primary_topic"] = None
        topics = {}

    # Uvijek napravi sve kolone, čak i ako parsing padne.
    for topic in TOPIC_KEYS:
        item = topics.get(topic, {}) if isinstance(topics, dict) else {}
        row[f"{topic}_mentioned"] = item.get("mentioned")
        row[f"{topic}_stance"] = item.get("stance")
        row[f"{topic}_confidence"] = item.get("confidence")
        row[f"{topic}_evidence"] = json.dumps(item.get("evidence", []), ensure_ascii=False)

    return row


In [22]:
# 9) TEST NA JEDNOM ČLANKU
# Pokreni ovo prije cijelog batcha da provjeriš da API key, model i JSON parsing rade.

def test_one_article(row_idx=0, prompt_name="prompt_2_evidence_first"):
    row = df.iloc[row_idx]
    prompt = render_prompt(PROMPTS[prompt_name], row)

    raw_response = call_openrouter(prompt)
    parsed = extract_json(raw_response)
    valid, errors = validate_annotation(parsed)

    print("VALID:", valid)
    print("ERRORS:", errors)
    print(json.dumps(parsed, ensure_ascii=False, indent=2))

    return parsed


# Odkomentariši za test:
# test_result = test_one_article(row_idx=0, prompt_name="prompt_2_evidence_first")


In [23]:
# 10) AGREEMENT I FINAL LABEL HELPERS

def _mode_or_none(series):
    values = [x for x in series.dropna().tolist() if x != ""]
    if not values:
        return None
    counts = pd.Series(values).value_counts()
    return counts.index[0]


def agreement_summary(flat_df):
    """
    Poredi promptove po article_id.
    Korisno da vidiš gdje se promptovi ne slažu.
    """
    if flat_df.empty:
        return pd.DataFrame()

    rows = []

    for article_id, g in flat_df.groupby("article_id"):
        item = {"article_id": article_id}

        for meta_col in ["portal", "datum", "naslov", "tema_sampling"]:
            if meta_col in g.columns:
                item[meta_col] = _mode_or_none(g[meta_col])

        rel_values = g["is_relevant"].dropna().astype(str).unique().tolist()
        item["is_relevant_values"] = "|".join(rel_values)
        item["is_relevant_agree"] = len(rel_values) == 1

        primary_values = g["primary_topic"].dropna().astype(str).unique().tolist()
        item["primary_topic_values"] = "|".join(primary_values)
        item["primary_topic_agree"] = len(primary_values) == 1

        for topic in TOPIC_KEYS:
            mentioned_values = g[f"{topic}_mentioned"].dropna().astype(str).unique().tolist()
            stance_values = g[f"{topic}_stance"].dropna().astype(str).unique().tolist()

            item[f"{topic}_mentioned_values"] = "|".join(mentioned_values)
            item[f"{topic}_mentioned_agree"] = len(mentioned_values) == 1

            item[f"{topic}_stance_values"] = "|".join(stance_values)
            item[f"{topic}_stance_agree"] = len(stance_values) == 1

        rows.append(item)

    return pd.DataFrame(rows)


def make_final_majority_labels(flat_df):
    """
    Jednostavna agregacija preko promptova:
    - za is_relevant i primary_topic uzima najčešću vrijednost,
    - za svaku temu uzima većinsku mentioned/stance vrijednost,
    - confidence je prosjek validnih confidence vrijednosti.
    """
    if flat_df.empty:
        return pd.DataFrame()

    rows = []

    for article_id, g in flat_df.groupby("article_id"):
        item = {"article_id": article_id}

        for meta_col in ["portal", "datum", "naslov", "tema_sampling"]:
            if meta_col in g.columns:
                item[meta_col] = _mode_or_none(g[meta_col])

        item["is_relevant_final"] = _mode_or_none(g["is_relevant"].astype("object"))
        item["primary_topic_final"] = _mode_or_none(g["primary_topic"].astype("object"))
        item["n_prompts"] = len(g)
        item["n_valid_json"] = int(g["valid_json"].sum())

        for topic in TOPIC_KEYS:
            mentioned_col = f"{topic}_mentioned"
            stance_col = f"{topic}_stance"
            confidence_col = f"{topic}_confidence"

            item[f"{topic}_mentioned_final"] = _mode_or_none(g[mentioned_col].astype("object"))
            item[f"{topic}_stance_final"] = _mode_or_none(g[stance_col].astype("object"))
            item[f"{topic}_confidence_mean"] = pd.to_numeric(g[confidence_col], errors="coerce").mean()

        rows.append(item)

    return pd.DataFrame(rows)


In [24]:
# 11) BATCH RUN

def read_existing_jsonl(path):
    records = []
    if not path.exists():
        return records

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    return records


def run_batch():
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    run_df = df.copy()
    if N_ROWS is not None:
        run_df = run_df.head(N_ROWS).copy()

    selected_prompts = {name: PROMPTS[name] for name in PROMPT_NAMES_TO_RUN}

    raw_jsonl_path = output_dir / "raw_prompt_outputs.jsonl"
    existing_records = read_existing_jsonl(raw_jsonl_path) if RESUME else []
    done_pairs = {
        (str(r.get("article_id")), str(r.get("prompt_name")))
        for r in existing_records
        if r.get("article_id") is not None and r.get("prompt_name") is not None
    }

    total_calls = len(run_df) * len(selected_prompts)
    skipped = 0
    new_calls = 0

    with open(raw_jsonl_path, "a", encoding="utf-8") as f_jsonl:
        call_idx = 0

        for _, row in run_df.iterrows():
            article_id = clean_text(row["article_id"])

            for prompt_name, template in selected_prompts.items():
                call_idx += 1
                pair = (article_id, prompt_name)

                if RESUME and pair in done_pairs:
                    skipped += 1
                    print(f"[{call_idx}/{total_calls}] SKIP article_id={article_id} | {prompt_name}")
                    continue

                new_calls += 1
                print(f"[{call_idx}/{total_calls}] article_id={article_id} | {prompt_name}")

                prompt = render_prompt(template, row)

                raw_response = ""
                parsed = None
                valid = False
                errors = []

                try:
                    raw_response = call_openrouter(prompt)
                    parsed = extract_json(raw_response)
                    valid, errors = validate_annotation(parsed)
                except Exception as e:
                    errors = [str(e)]

                record = {
                    "article_id": article_id,
                    "prompt_name": prompt_name,
                    "valid_json": valid,
                    "errors": errors,
                    "parsed": parsed,
                    "raw_response": raw_response,
                }

                f_jsonl.write(json.dumps(record, ensure_ascii=False) + "\n")
                f_jsonl.flush()

                time.sleep(SLEEP_BETWEEN)

    print(f"\nBatch završen. Novi pozivi: {new_calls}, preskočeno zbog RESUME: {skipped}")
    print("Raw JSONL:", raw_jsonl_path)

    return build_output_tables(raw_jsonl_path, run_df, output_dir)


def build_output_tables(raw_jsonl_path, source_df, output_dir):
    records = read_existing_jsonl(raw_jsonl_path)

    source_by_id = {
        clean_text(row["article_id"]): row
        for _, row in source_df.iterrows()
    }

    flat_rows = []
    for record in records:
        article_id = clean_text(record.get("article_id", ""))
        source_row = source_by_id.get(article_id, {})

        flat_rows.append(
            flatten_result(
                article_id=article_id,
                prompt_name=record.get("prompt_name", ""),
                parsed=record.get("parsed"),
                valid=record.get("valid_json", False),
                errors=record.get("errors", []),
                raw_response=record.get("raw_response", ""),
                source_row=source_row,
            )
        )

    flat_df = pd.DataFrame(flat_rows)

    flat_csv_path = output_dir / "flat_prompt_results.csv"
    flat_df.to_csv(flat_csv_path, index=False, encoding="utf-8-sig")

    summary_df = agreement_summary(flat_df)
    summary_csv_path = output_dir / "prompt_agreement_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

    final_df = make_final_majority_labels(flat_df)
    final_csv_path = output_dir / "final_majority_labels.csv"
    final_df.to_csv(final_csv_path, index=False, encoding="utf-8-sig")

    print("\nOutput fajlovi:")
    print("Flat CSV:", flat_csv_path)
    print("Agreement summary CSV:", summary_csv_path)
    print("Final majority labels CSV:", final_csv_path)

    if len(flat_df):
        print("\nValid JSON po promptu:")
        display(flat_df.groupby("prompt_name")["valid_json"].mean())

        print("\nRelevant count po promptu:")
        display(flat_df.groupby("prompt_name")["is_relevant"].sum())

    return flat_df, summary_df, final_df


In [ ]:
# 12) POKRETANJE CIJELOG BATCHA

# Za prvi test preporuka:
# 1) ostavi N_ROWS = 10
# 2) pokreni ovu ćeliju
# 3) ako je sve uredu, vrati se gore i stavi N_ROWS = None za cijeli dataset

flat_df, summary_df, final_df = run_batch()

display(flat_df.head())
display(summary_df.head())
display(final_df.head())


[1/4000] article_id=12177 | prompt_2_evidence_first
[2/4000] article_id=21123 | prompt_2_evidence_first
[3/4000] article_id=5583 | prompt_2_evidence_first
[4/4000] article_id=17855 | prompt_2_evidence_first
[5/4000] article_id=34982 | prompt_2_evidence_first
[6/4000] article_id=43091 | prompt_2_evidence_first
[7/4000] article_id=46383 | prompt_2_evidence_first
[8/4000] article_id=1517 | prompt_2_evidence_first
[9/4000] article_id=25473 | prompt_2_evidence_first
[10/4000] article_id=19637 | prompt_2_evidence_first
[11/4000] article_id=43790 | prompt_2_evidence_first
[12/4000] article_id=28253 | prompt_2_evidence_first
[13/4000] article_id=9978 | prompt_2_evidence_first
[14/4000] article_id=48152 | prompt_2_evidence_first
[15/4000] article_id=35599 | prompt_2_evidence_first
[16/4000] article_id=29885 | prompt_2_evidence_first
[17/4000] article_id=19358 | prompt_2_evidence_first
[18/4000] article_id=4314 | prompt_2_evidence_first
[19/4000] article_id=30107 | prompt_2_evidence_first
[20/40

In [ ]:
flat_df

In [ ]:
import pandas as pd
flat_df = pd.read_csv("openrouter_prompt_tests/flat_prompt_results.csv")
df = pd.read_csv("novi_sample_za_llm.csv")

In [ ]:
df[df["article_id"]==107708]

In [ ]:
flat_df

In [ ]:
article_text_cols = df[["article_id", "SADRZAJ"]].copy()
article_text_cols["article_id"] = article_text_cols["article_id"].astype(str)

flat_df["article_id"] = flat_df["article_id"].astype(str)

flat_df = flat_df.merge(
    article_text_cols,
    on="article_id",
    how="left"
)

display(flat_df.head())

In [ ]:
# Premjesti kolonu sadrzaj da bude druga kolona po redu

col = "sadrzaj" if "sadrzaj" in flat_df.columns else "SADRZAJ"

cols = flat_df.columns.tolist()
cols.insert(1, cols.pop(cols.index(col)))

flat_df = flat_df[cols]

display(flat_df)

In [ ]:
flat_df.to_csv("novi_dataset_stance_4000.csv", index=False, encoding="utf-8-sig")